<a href="https://colab.research.google.com/github/Sunny-Mansoor/BlockQuest/blob/main/ELT%20Pipeline%20with%20Kafka%20and%20PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Download and extract Kafka
!wget https://archive.apache.org/dist/kafka/3.5.0/kafka_2.12-3.5.0.tgz
!tar -xzf kafka_2.12-3.5.0.tgz

# 2. Start Zookeeper (Kafka's manager) in the background
import os
os.system("kafka_2.12-3.5.0/bin/zookeeper-server-start.sh -daemon kafka_2.12-3.5.0/config/zookeeper.properties")

# 3. Start the Kafka Broker in the background
os.system("kafka_2.12-3.5.0/bin/kafka-server-start.sh -daemon kafka_2.12-3.5.0/config/server.properties")

print("Kafka is now running on localhost:9092")

--2026-10-26 17:01:44--  https://archive.apache.org/dist/kafka/3.5.0/kafka_2.12-3.5.0.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 107000763 (102M) [application/x-gzip]
Saving to: ‘kafka_2.12-3.5.0.tgz.2’

kafka_2.12-3.5.0.tg 100%[===================>] 102.04M   175KB/s    in 5m 36s  

2026-10-26 17:07:20 (311 KB/s) - ‘kafka_2.12-3.5.0.tgz.2’ saved [107000763/107000763]

Kafka is now running on localhost:9092


In [3]:
!pip install -q kafka-python

from kafka import KafkaProducer
import json
import time
import random

# Connect to the local Kafka broker we just started
producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

locations = ["Downtown", "Airport", "Suburb", "Financial District", "Stadium"]

print("Starting RideGo Live Simulator...")

# Generate 50 simulated ride requests
for _ in range(50):
    ride_event = {
        "ride_id": random.randint(10000, 99999),
        "pickup_location": random.choice(locations),
        "surge_multiplier": round(random.uniform(1.0, 3.5), 1),
        "fare_estimate": round(random.uniform(15.0, 80.0), 2),
        "timestamp": time.time()
    }

    # Ingest into Kafka
    producer.send('ride_requests', ride_event)
    print(f"Ingested: {ride_event}")
    time.sleep(0.3) # Simulate a delay between rides

print("✅ Simulation complete. Data is safely stored in Kafka.")

Starting RideGo Live Simulator...
Ingested: {'ride_id': 98863, 'pickup_location': 'Stadium', 'surge_multiplier': 3.0, 'fare_estimate': 55.17, 'timestamp': 1793034451.943243}
Ingested: {'ride_id': 93277, 'pickup_location': 'Stadium', 'surge_multiplier': 1.8, 'fare_estimate': 24.21, 'timestamp': 1793034452.3458734}
Ingested: {'ride_id': 54033, 'pickup_location': 'Stadium', 'surge_multiplier': 2.8, 'fare_estimate': 33.65, 'timestamp': 1793034452.6466117}
Ingested: {'ride_id': 63239, 'pickup_location': 'Financial District', 'surge_multiplier': 3.0, 'fare_estimate': 68.08, 'timestamp': 1793034452.9484322}
Ingested: {'ride_id': 50821, 'pickup_location': 'Financial District', 'surge_multiplier': 1.7, 'fare_estimate': 40.2, 'timestamp': 1793034453.249482}
Ingested: {'ride_id': 43553, 'pickup_location': 'Financial District', 'surge_multiplier': 2.4, 'fare_estimate': 75.57, 'timestamp': 1793034453.55113}
Ingested: {'ride_id': 51360, 'pickup_location': 'Financial District', 'surge_multiplier': 1.

In [1]:
# 1. Update and install Java
!apt-get update -qq
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

# 2. Install PySpark (pinned to 3.5.0) and findspark
!pip install -q pyspark==3.5.0 findspark

# 3. Configure robust Environment Variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"  # Fixes the Gateway Exited error!

# 4. Initialize Spark paths safely
import findspark
findspark.init()

print("✅ Java, PySpark, and Py4J Gateway are securely configured.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.
✅ Java, PySpark, and Py4J Gateway are securely configured.


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, DoubleType, IntegerType

# 1. Initialize Spark (with the Kafka connector package)
spark = SparkSession.builder \
    .appName("RideGo_Surge_Pipeline") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

# 2. EXTRACT: Read the raw streaming data from Kafka
kafka_stream_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "ride_requests") \
    .option("startingOffsets", "earliest") \
    .load()

# 3. Define the schema to map our JSON data
schema = StructType() \
    .add("ride_id", IntegerType()) \
    .add("pickup_location", StringType()) \
    .add("surge_multiplier", DoubleType()) \
    .add("fare_estimate", DoubleType()) \
    .add("timestamp", DoubleType())

# 4. TRANSFORM: Parse the binary Kafka data into readable columns
parsed_df = kafka_stream_df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

# 5. TRANSFORM: Filter for rides with a surge multiplier greater than 2.0
high_surge_rides = parsed_df.filter(col("surge_multiplier") > 2.0)

# 6. LOAD: Write the transformed stream into an in-memory table
query = high_surge_rides.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("high_surge_dashboard") \
    .start()

print("✅ PySpark Structured Streaming pipeline is active and listening.")

✅ PySpark Structured Streaming pipeline is active and listening.


In [5]:
# Query the in-memory table populated by our streaming pipeline
print("🚨 RIDEGO: HIGH SURGE LOCATIONS DETECTED 🚨")
spark.sql("SELECT ride_id, pickup_location, surge_multiplier, fare_estimate FROM high_surge_dashboard ORDER BY surge_multiplier DESC").show(truncate=False)

🚨 RIDEGO: HIGH SURGE LOCATIONS DETECTED 🚨
+-------+------------------+----------------+-------------+
|ride_id|pickup_location   |surge_multiplier|fare_estimate|
+-------+------------------+----------------+-------------+
|54500  |Airport           |3.5             |18.0         |
|83468  |Downtown          |3.4             |74.14        |
|30707  |Downtown          |3.4             |29.29        |
|56575  |Downtown          |3.4             |23.35        |
|48816  |Suburb            |3.3             |79.39        |
|86242  |Airport           |3.3             |30.03        |
|92988  |Financial District|3.3             |45.25        |
|89682  |Financial District|3.3             |51.7         |
|41901  |Stadium           |3.3             |78.78        |
|40881  |Downtown          |3.3             |67.13        |
|67258  |Airport           |3.2             |67.01        |
|58197  |Suburb            |3.2             |34.71        |
|27568  |Financial District|3.2             |24.22        